In [1]:
import xarray as xr
import numpy as np

# =====================================================
# 1. LOAD DATASET
# =====================================================

input_file = "GLORY_3D_surface_monthly_1993_2025.nc"

print("Loading dataset...")
ds = xr.open_dataset(input_file)


# =====================================================
# 2. METRIC CONVERSION
# =====================================================

print("Preparing metric conversion factors...")

deg_to_m = 111320.0

cos_lat = np.cos(np.deg2rad(ds.latitude))

dx_per_degree = deg_to_m * cos_lat
dy_per_degree = deg_to_m

# =====================================================
# 3. VELOCITY DERIVATIVES
# =====================================================

print("Computing velocity derivatives...")

du_dx = (
    ds["uo"].differentiate("longitude")
    / dx_per_degree
)

du_dy = (
    ds["uo"].differentiate("latitude")
    / dy_per_degree
)

dv_dx = (
    ds["vo"].differentiate("longitude")
    / dx_per_degree
)

dv_dy = (
    ds["vo"].differentiate("latitude")
    / dy_per_degree
)

# =====================================================
# 4. SSH DERIVATIVES
# =====================================================

print("Computing SSH derivatives...")

dh_dx = (
    ds["zos"].differentiate("longitude")
    / dx_per_degree
)

dh_dy = (
    ds["zos"].differentiate("latitude")
    / dy_per_degree
)

# =====================================================
# 5. VORTICITY
# =====================================================

print("Calculating vorticity...")

ds["vorticity"] = dv_dx - du_dy

ds["vorticity"].attrs = {
    "long_name": "Relative Vorticity",
    "units": "s^-1"
}

# =====================================================
# 6. OKUBO-WEISS
# =====================================================

print("Calculating Okubo-Weiss parameter...")

normal_strain = du_dx - dv_dy

shear_strain = dv_dx + du_dy

ds["okubo_weiss"] = (
    normal_strain**2
    + shear_strain**2
    - ds["vorticity"]**2
)

ds["okubo_weiss"].attrs = {
    "long_name": "Okubo Weiss Parameter",
    "units": "s^-2"
}

# =====================================================
# 7. SSH GRADIENT MAGNITUDE
# =====================================================

print("Calculating SSH gradient magnitude...")

ds["grad_ssh"] = np.sqrt(
    dh_dx**2 +
    dh_dy**2
)

ds["grad_ssh"].attrs = {
    "long_name": "Sea Surface Height Gradient",
    "units": "m/m"
}

# =====================================================
# 8. KEEP VARIABLES
# =====================================================

vars_to_keep = [

    "zos",
    "so",
    "thetao",
    "uo",
    "vo",
    "mlotst",

    "vorticity",
    "okubo_weiss",
    "grad_ssh"
]

ds_final = ds[vars_to_keep]

# =====================================================
# 9. SAVE NETCDF
# =====================================================

output_file = "monthly_glorys_with_dynamics.nc"

print(f"Saving {output_file} ...")

encoding = {
    var: {
        "zlib": True,
        "complevel": 4
    }
    for var in ds_final.data_vars
}

ds_final.to_netcdf(
    output_file,
    encoding=encoding
)

print("Done.")
print(f"Saved: {output_file}")

Loading dataset...
Preparing metric conversion factors...
Computing velocity derivatives...
Computing SSH derivatives...
Calculating vorticity...
Calculating Okubo-Weiss parameter...
Calculating SSH gradient magnitude...
Saving monthly_glorys_with_dynamics.nc ...
Done.
Saved: monthly_glorys_with_dynamics.nc
